In [1]:
import os
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

IMG_ROOT = '../data/Images'
IMG_SIZE = (128, 128)
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',
}

## Load all images into memory

EfficientNetB0 uses `preprocess_input` to scale pixels to [-1, 1].

In [2]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32')
    return preprocess_input(arr)

records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)

print(f'Loading {len(df)} images...')
X = np.array([load_image(p) for p in df['path']])
print(f'Loaded. Array shape: {X.shape}  Memory: {X.nbytes / 1e9:.2f} GB')

le = LabelEncoder()
y = le.fit_transform(df['label'])
print('Classes:', le.classes_)

Loading 4869 images...
Loaded. Array shape: (4869, 128, 128, 3)  Memory: 0.96 GB
Classes: ['negative' 'neutral' 'positive']


## Train/val/test split

In [3]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

Train: 3116  Val: 779  Test: 974


## Tune classification head on validation set

EfficientNetB0 base is frozen (weights='imagenet'). Tune dense head units.

In [4]:
def build_model(dense_units):
    base = EfficientNetB0(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = False
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(dense_units, activation='relu', kernel_regularizer=l2(1e-4)),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

t0 = time.time()
units_options = [128, 256]
val_results = []
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

for units in units_options:
    model = build_model(units)
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    val_results.append({'dense_units': units, 'val_acc': val_acc})
    print(f'dense_units={units:<4}  Val Accuracy: {val_acc:.3f}')

best_units = max(val_results, key=lambda x: x['val_acc'])['dense_units']
print(f'\nBest dense_units: {best_units}')

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
dense_units=128   Val Accuracy: 0.368
dense_units=256   Val Accuracy: 0.353

Best dense_units: 128


## Retrain on full train+val, evaluate on test

Uses `validation_split=0.2` on X_trainval rather than the fixed val set, giving
EarlyStopping a fresh signal not seen during hyperparameter tuning.

In [5]:
best_model = build_model(best_units)
best_model.fit(
    X_trainval, y_trainval,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)
runtime = time.time() - t0

y_pred_enc = np.argmax(best_model.predict(X_test), axis=1)
y_pred = le.inverse_transform(y_pred_enc)
y_test_labels = le.inverse_transform(y_test)

print(f'\nAccuracy: {accuracy_score(y_test_labels, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test_labels, y_pred))

report = classification_report(y_test_labels, y_pred, output_dict=True)
meta = {
    'model': 'cnn_efficientnet',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
best_model.save('../models/images/fits/cnn_efficientnet.keras')
with open('../models/images/json/cnn_efficientnet_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')

Epoch 1/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 38s 279ms/step - accuracy: 0.3434 - loss: 1.2223 - val_accuracy: 0.3877 - val_loss: 1.1354
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 24s 242ms/step - accuracy: 0.3662 - loss: 1.1764 - val_accuracy: 0.3877 - val_loss: 1.1255
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 24s 245ms/step - accuracy: 0.4018 - loss: 1.1240 - val_accuracy: 0.3915 - val_loss: 1.1206
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 20s 206ms/step - accuracy: 0.4101 - loss: 1.1085 - val_accuracy: 0.4018 - val_loss: 1.1178
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 20s 199ms/step - accuracy: 0.4397 - loss: 1.0750 - val_accuracy: 0.3979 - val_loss: 1.1160
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 19s 198ms/step - accuracy: 0.4769 - loss: 1.0425 - val_accuracy: 0.3954 - val_loss: 1.1161
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 215ms/step - accuracy: 0.4814 - loss: 1.0346 - val_accuracy: 0.3902 - val_loss: 1.1143
Epoch 8/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 22s 230ms/step - accuracy: 0.5074 - loss: 1.0151 - 